### 1. Import Libraries and Load Data

In [ ]:
import ast
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset


dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])
df['job_skills'] = df['job_skills'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)


df_analyst = df[df['job_title_short'] == 'Data Analyst']

### 2. Explode Skills

In [ ]:
df_skills = df_analyst.copy()
df_skills['job_skills'] = df_skills['job_skills'].apply(lambda x: x if isinstance(x, list) else [])
df_skills_exploded = df_skills.explode('job_skills')

### 3. Count Skills per Job Title

In [ ]:
df_skills_count = (
    df_skills_exploded.groupby(['job_skills', 'job_title_short'])
    .size()
    .reset_index(name='skill_count')
)
df_skills_count = df_skills_count.sort_values(by='skill_count', ascending=False)

### 4. Create List of Top Roles

In [ ]:
job_titles = df_skills_count['job_title_short'].unique().tolist()
job_titles = job_titles[:3]

### 5. Plot Skill Counts

In [ ]:
fig, ax = plt.subplots(len(job_titles), 1, figsize=(8, 12), sharex=True)
sns.set_theme(style='ticks')

for i, job_title in enumerate(job_titles):
    data = df_skills_count[df_skills_count['job_title_short'] == job_title].head(5)
    sns.barplot(
        data=data,
        x='skill_count',
        y='job_skills',
        ax=ax[i],
        hue='skill_count',
        palette='dark:b_r',
        legend=False
    )
    ax[i].set_title(job_title)
    ax[i].set_ylabel('')
    ax[i].set_xlabel('Number of Jobs')

plt.tight_layout()
plt.show()

In [ ]:
### 6. Convert Counts to Percentages

df_job_count = df_analyst['job_title_short'].value_counts().reset_index(name='jobs_total')

df_skills_percentage = pd.merge(df_skills_count, df_job_count, on='job_title_short', how='left')
df_skills_percentage['skill_percent'] = (df_skills_percentage['skill_count'] / df_skills_percentage['jobs_total']) * 100

### 7. Plot Skill Percentage

In [ ]:
df_job_count = df_analyst['job_title_short'].value_counts().reset_index(name='jobs_total')

df_skills_percentage = pd.merge(
    df_skills_count,
    df_job_count,
    on='job_title_short',
    how='left'
)
df_skills_percentage['skill_percent'] = (
    df_skills_percentage['skill_count'] / df_skills_percentage['jobs_total']
) * 100

fig, ax = plt.subplots(len(job_titles), 1, figsize=(8, 12), sharex=True)
sns.set_theme(style='ticks')

if len(job_titles) == 1:
    ax = [ax]

for i, job_title in enumerate(job_titles):
    data = df_skills_percentage[df_skills_percentage['job_title_short'] == job_title].head(5)
    sns.barplot(
        data=data,
        x='skill_percent',
        y='job_skills',
        ax=ax[i],
        hue='skill_percent',
        palette='dark:b_r',
        legend=False
    )
    ax[i].set_title(job_title)
    ax[i].set_ylabel('')
    ax[i].set_xlabel('Likelihood of Skills Required (%)')
    ax[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x)}%'))

plt.tight_layout()
plt.show()